# Modélisation finale à J30

**Objectif :** comparer plusieurs modèles afin de prédire, à J30 depuis le début du programme, l’abandon ultérieur enregistré des inscriptions éligibles (`churn`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Ouvrir ce notebook depuis la racine du projet ou notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay, brier_score_loss
from src.preprocessing import BASE_NUM, CAT, FEATURES, FLAGS, TARGET, load_j30, split, prepare
from src.modeling import (
    compare_models, describe, forest, threshold, scores, diagnostics,
    validation_importance, evaluate_temporal,
)
from src.config import J30_RESULTS

## Préparation des données

### Chargement et éligibilité

Les inscriptions du panel J30 sont réparties par étudiant (graine 10) : 25 % des groupes au test, puis 20 % des groupes restants à la validation. Les inscriptions sans date de début, débutant après J30 ou annulées au plus tard à J30 sont ensuite exclues de chaque partition.

L’intensité d’activité et le nombre de soumissions sont calculés sur J1–J30, et la récence sur J0–J30. Les moyennes cumulées incluent les soumissions antérieures à la rentrée retenues lors du nettoyage. Les comptes utilisent les jours relatifs entiers, tandis que les moyennes utilisent la date et l’heure exactes de J30. Les notes sont supposées disponibles à la soumission, de même que les échéances utilisées pour calculer les retards.

In [ ]:
data, evaluations = load_j30()
partitions = split(data)
train, val, test = (prepare(part, evaluations) for part in partitions)
student_groups = [set(part.id_etudiant) for part in (train, val, test)]
assert not (student_groups[0] & student_groups[1])
assert not (student_groups[0] & student_groups[2])
assert not (student_groups[1] & student_groups[2])

x_tr, y_tr = train[FEATURES], train[TARGET]
x_val, y_val = val[FEATURES], val[TARGET]
x_test, y_test = test[FEATURES], test[TARGET]

### Répartition des observations

In [ ]:
population = pd.DataFrame({
    "entraînement": describe(train), "validation": describe(val), "test": describe(test)
}).T
population["exclusions_J30"] = [
    len(old) - len(new) for old, new in zip(partitions, (train, val, test))
]
population

### Variables et prétraitement

Les moyennes de notes et de retards indisponibles sont imputées par la médiane et accompagnées d’indicateurs binaires. Le niveau scolaire est imputé par le mode puis encodé en indicatrices. Ces transformations sont apprises sur l’entraînement, puis réajustées sur l’ensemble entraînement et validation pour le modèle final.

Les variables numériques sont standardisées pour la régression logistique, le SVM et le gradient boosting, mais pas pour les arbres ni la forêt.

In [ ]:
print("Variables, dans l’ordre :", FEATURES)
print("Indicateurs de manque, entraînement :")
print(train[FLAGS].sum())
missingness = {column: int(pd.concat([train, val, test])[column].sum()) for column in FLAGS}
print("Valeurs manquantes dans la population J30 éligible :")
print(pd.Series(missingness))

## Modèles de prédiction

L’**Average Precision (AP)** correspond à la moyenne des précisions pondérée par les incréments de rappel. Elle diffère de l’aire trapézoïdale sous la courbe précision-rappel.

### Régression logistique

Paramètres et résultats sur la validation, sans pondération des classes :

In [ ]:
comparison = compare_models(train, val)
comparison["logistic"]["parameters"]

Scores sur validation :

In [ ]:
pd.Series(comparison["logistic"]["metrics"], name="validation")

### Arbre de décision

In [ ]:
comparison["tree_default"]["parameters"]

In [ ]:
pd.Series(comparison["tree_default"]["metrics"], name="validation")

La grille croise `max_depth = [3, 5, 7, 10, None]` et `min_samples_leaf = [1, 5, 10, 20]`. Le meilleur arbre est choisi par AP de validation, dans l’ordre de cette grille en cas d’égalité.

In [ ]:
print(comparison["tree_tuned"]["parameters"])
pd.Series(comparison["tree_tuned"]["metrics"], name="validation")

### SVM

In [ ]:
comparison["svm"]["parameters"]

In [ ]:
pd.Series(comparison["svm"]["metrics"], name="validation")

In [ ]:
pd.DataFrame({
    name: comparison[name]["metrics"]
    for name in ("logistic_balanced", "tree_balanced", "svm_balanced")
}).T

In [ ]:
forest_model = forest()
forest_model.fit(x_tr, y_tr)
validation_probabilities = forest_model.predict_proba(x_val)[:, 1]
forest_validation = scores(y_val, validation_probabilities)
print("Forêt :", forest_model.named_steps["model"].get_params())
pd.DataFrame({
    "Random Forest": forest_validation,
    "Gradient Boosting": comparison["gradient_boosting"]["metrics"],
}).T

## Comparaison des modèles

La forêt retenue comporte 300 arbres, `min_samples_leaf=10`, `max_features="sqrt"`, `random_state=10` et aucune pondération des classes.

In [ ]:
results = pd.DataFrame({name: result["metrics"] for name, result in comparison.items()}).T
results.loc["random_forest"] = pd.Series(forest_validation)
results.sort_values("ap", ascending=False)

In [ ]:
sns.barplot(data=results.reset_index(), x="index", y="ap", color="steelblue")
plt.axhline(y_val.mean(), color="red", linestyle="--", label="Prévalence de validation")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Modèle")
plt.ylabel("Average Precision")
plt.legend()
plt.tight_layout()
plt.show()

## Seuil et évaluation finale

Le seuil maximise le rappel sous une contrainte de précision d’au moins 30 % sur validation. En cas d’égalité, la précision puis le seuil les plus élevés sont privilégiés. Le seuil est ensuite conservé après réentraînement de la forêt sur l’ensemble entraînement et validation.

In [ ]:
validation_diagnostics = diagnostics(y_val, validation_probabilities)
pd.DataFrame(validation_diagnostics["threshold_grid"])

In [ ]:
best_threshold = threshold(y_val, validation_probabilities)
if best_threshold is None:
    raise ValueError("Aucun seuil ne satisfait la précision minimale de validation.")
validation_metrics = scores(y_val, validation_probabilities, best_threshold)
grid = pd.DataFrame(validation_diagnostics["threshold_grid"])
fig, ax = plt.subplots()
ax.plot(grid["threshold"], grid["precision"], label="Précision")
ax.plot(grid["threshold"], grid["recall"], label="Rappel")
ax.axhline(0.30, color="grey", linestyle="--", label="Précision minimale")
ax.axvline(best_threshold, color="grey", linestyle=":", label="Seuil retenu")
ax.scatter([best_threshold] * 2, [validation_metrics["precision"], validation_metrics["recall"]])
ax.set(xlabel="Seuil de classification", ylabel="Proportion", ylim=(0, 1.03))
ax.legend()
plt.show()
pd.Series(validation_metrics)

### Traitement des moyennes manquantes

Deux forêts de paramètres identiques sont comparées sur les mêmes partitions éligibles. La première utilise les moyennes du panel longitudinal, encodées à zéro en cas d’absence ; la seconde utilise les moyennes au seuil temporel exact de J30, l’imputation médiane et les indicateurs d’absence. La comparaison n’isole donc pas le seul effet de l’imputation.

Le traitement par imputation et indicateurs est retenu pour distinguer les valeurs manquantes des zéros observés.

In [ ]:
zero_train, zero_val = [
    original.loc[original.id_inscription.isin(eligible.id_inscription)]
    for original, eligible in zip(partitions[:2], (train, val))
]
for original, eligible in zip((zero_train, zero_val), (train, val)):
    assert original[["id_inscription", TARGET]].equals(eligible[["id_inscription", TARGET]])
    assert original[BASE_NUM + CAT].equals(eligible[BASE_NUM + CAT].assign(
        avg_note_cum=original.avg_note_cum, avg_lateness_cum=original.avg_lateness_cum,
    ))

zero_model = forest(include_missingness=False).fit(zero_train[BASE_NUM + CAT], zero_train[TARGET])
assert zero_model.named_steps["model"].get_params() == forest_model.named_steps["model"].get_params()
zero_probabilities = zero_model.predict_proba(zero_val[BASE_NUM + CAT])[:, 1]
zero_threshold = threshold(zero_val[TARGET], zero_probabilities)
if zero_threshold is None:
    raise ValueError("Aucun seuil admissible pour la variante à zéro.")
zero_encoding_validation = scores(zero_val[TARGET], zero_probabilities, zero_threshold)
pd.DataFrame({
    "Encodage à zéro": zero_encoding_validation,
    "Médiane et indicateurs": validation_metrics,
}).loc[["ap", "brier", "log_loss", "threshold", "precision", "recall"]]


In [ ]:
final_model = clone(forest_model).fit(
    pd.concat([x_tr, x_val]), pd.concat([y_tr, y_val])
)
test_probabilities = final_model.predict_proba(x_test)[:, 1]
test_metrics = scores(y_test, test_probabilities, best_threshold)
print(pd.Series(test_metrics))
baseline_brier = brier_score_loss(
    y_test, [pd.concat([y_tr, y_val]).mean()] * len(y_test)
)
print("Brier du prédicteur constant (prévalence entraînement + validation) :", baseline_brier)
ConfusionMatrixDisplay.from_predictions(
    y_test, test_probabilities >= best_threshold, display_labels=[0, 1], cmap="Blues"
)
plt.show()

## Calibration et gains cumulés

On examine la calibration sans recalibrer le modèle. Les classes de probabilité ont une largeur de 0,1 ; celles de moins de 20 observations ne sont pas affichées. Les gains répartissent proportionnellement les ex æquo au seuil de sélection, afin de ne pas dépendre de l’ordre des lignes.

In [ ]:
test_diagnostics = diagnostics(y_test, test_probabilities)
fig, ax = plt.subplots()
for label, summary in [("Validation", validation_diagnostics), ("Test", test_diagnostics)]:
    bins = pd.DataFrame(summary["calibration_bins"])
    if not bins.empty:
        ax.plot(bins["mean_probability"], bins["observed_rate"], marker="o", label=label)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set(xlabel="Probabilité moyenne", ylabel="Fréquence observée", title="Calibration")
ax.legend()
plt.show()

In [ ]:
gains = pd.DataFrame(test_diagnostics["gains"])
plt.plot(gains["fraction"], gains["recall"], marker="o", label="Forêt — test")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Sélection aléatoire")
plt.xlabel("Part des inscriptions ciblées")
plt.ylabel("Part des abandons retrouvés")
plt.legend()
plt.show()
gains

## Importance par permutation

L’importance correspond à la diminution moyenne de l’AP sur validation après permutation de chaque variable (15 répétitions, graine 10). Elle mesure la contribution prédictive, sans interprétation causale.

In [ ]:
importance = pd.DataFrame(validation_importance(train, val))
ranked_importance = importance.sort_values("mean_ap_decrease")
fig, ax = plt.subplots()
ax.barh(ranked_importance["feature"], ranked_importance["mean_ap_decrease"],
        xerr=ranked_importance["sd"], capsize=3)
ax.axvline(0, color="grey", linewidth=1)
ax.set_xlabel("Diminution moyenne de l’AP après permutation")
fig.tight_layout()
plt.show()
importance.sort_values("mean_ap_decrease", ascending=False)

## Validation temporelle rétrospective

L’année 2023/2024 sert à l’entraînement et à la validation, et 2024/2025 au test. Les étudiants présents dans les deux années sont retirés de la première. Un seuil propre à cette expérience est sélectionné sur la validation selon la même contrainte de précision.

In [ ]:
temporal = evaluate_temporal(pd.concat([train, val, test]))
if "test_metrics" in temporal:
    print("Étudiants communs retirés de l’année antérieure :",
          temporal["shared_students_removed_from_earlier_year"])
    print(pd.Series(temporal["test_metrics"]))
else:
    print(temporal["note"])

## Résultats agrégés

Les effectifs, paramètres, métriques et diagnostics sont regroupés dans `output/j30_results.json`.

In [ ]:
experiment_results = {
    "population": describe(pd.concat([train, val, test])),
    "partitions": {"train": describe(train), "validation": describe(val), "test": describe(test)},
    "excluded_per_partition": [len(before) - len(after) for before, after in zip(partitions, (train, val, test))],
    "missingness": missingness,
    "zero_encoding_validation": zero_encoding_validation,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "baseline_brier": float(baseline_brier),
    "validation_comparison": {
        **comparison,
        "random_forest": {"metrics": forest_validation, "parameters": forest_model.named_steps["model"].get_params()},
    },
    "forest_parameters": final_model.named_steps["model"].get_params(),
    "validation_diagnostics": validation_diagnostics,
    "test_diagnostics": test_diagnostics,
    "validation_permutation_importance": importance.to_dict(orient="records"),
    "year_holdout": temporal,
}
J30_RESULTS.parent.mkdir(parents=True, exist_ok=True)
J30_RESULTS.write_text(json.dumps(experiment_results, indent=2, allow_nan=False), encoding="utf-8")
print("Résultats agrégés enregistrés dans", J30_RESULTS.name)

## Conclusion et limites

La forêt aléatoire fournit un classement informatif, mais sa précision reste modeste au seuil retenu. La priorisation des inscriptions selon leur score peut aider à organiser le suivi pédagogique, sans établir l’efficacité d’une intervention.

L’abandon enregistré peut être postérieur au désengagement réel. Le jeu de données exclut les inscriptions sans activité sur toute la période, ce qui peut introduire un biais de sélection à J30. Les dates officielles de fin de programme ne sont pas connues, et les dates de disponibilité des notes ne sont pas documentées séparément. L’exploration du panel a porté sur l’ensemble des inscriptions ; ni le test groupé ni le test par année ne constituent une validation externe entièrement indépendante.